In [ ]:
import os 
from glob import glob
import SimpleITK as sitk
import numpy as np
import datasets.path as path
from segment_anything import sam_model_registry
from segment_anything.utils.transforms import ResizeLongestSide
from skimage import transform, segmentation, io
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from utils import min_max_norm_3dimg
import json
from pre_embeddings import sam_preprocess

In [ ]:
# training and testing saving dataset path prefix
prefix = './datasets/RAINE_organ_51'
task = 'MRI_LeftKidney'
# label id
# left kidney: 1,
# right kidney: 2,
# pancreas: 3,
# background: 0,
label_id = 1
# image size
image_size = 256
#SAM MODEL TYPE 
sam_model_type = 'vit_b'
# SAM checkpoint
checkpoint = './checkpoints/SAM/sam_vit_b_01ec64.pth'
# device 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# load data paths
images = sorted(glob(os.path.join(path.RAINE_ORGAN_IMAGES_51, '*.nii.gz')))
gts = sorted(glob(os.path.join(path.RAINE_ORGAN_MANUAL_GTS_51, '*.nii.gz')))
names = [os.path.basename(image).split('/')[-1] for image in images]
# split data into train, test
np.random.seed(2023)
np.random.shuffle(names)
train_names = sorted(names[:int(len(names)*0.8)]) # 40
test_names = sorted(names[int(len(names)*0.8):]) # 11
# prepare the save path
save_path_tr = os.path.join(prefix, task, 'train')
save_path_ts = os.path.join(prefix, task, 'test')
# save data ti train nad test folder
os.makedirs(save_path_tr, exist_ok=True)
os.makedirs(save_path_ts, exist_ok=True)


In [ ]:
from torch.utils.data import DataLoader, Dataset
class NpzDataset(Dataset): 
 
    def __init__(self, sample_pool_path):
        self.npz_files = sorted(sample_pool_path) 
        # print(self.npz_files[0])
        img = [sitk.ReadImage(f) for f in self.npz_files]
        img_array = [sitk.GetArrayFromImage(i) for i in img]
        self.imgs = np.vstack(img_array)
    
    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, index):
        # inputs = []
        image = self.imgs[index]
        # for i in range(image.shape[0]):
        sam_model = sam_model_registry[sam_model_type](checkpoint=checkpoint).to(device)
        sam_transform = ResizeLongestSide(sam_model.image_encoder.img_size)
        print(f"{image.shape=}")
        image = np.uint8(np.repeat(image[:,:,None], 3, axis=-1))
        print(f"after 3c {image.shape=}")
        resize_img = sam_transform.apply_image(image)
        
        resize_img_tensor = torch.as_tensor(resize_img.transpose(2, 0, 1)).to(device)
        input_image = sam_model.preprocess(resize_img_tensor) # (1, 3, 1024, 1024)
        return torch.tensor(input_image).float()

In [ ]:
# from utils import NpzDataset
from torch.utils.data import DataLoader, Dataset

sam_model = sam_model_registry[sam_model_type](checkpoint=checkpoint).to(device)

images = sorted(glob(os.path.join(path.RAINE_ORGAN_IMAGES_51, '*.nii.gz')))
gts = sorted(glob(os.path.join(path.RAINE_ORGAN_MANUAL_GTS_51, '*.nii.gz')))

dataset = NpzDataset([images[0]])
# batch_size = len(dataset)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False) # make sure batch size is larger than sample slices
for bimg in dataloader:
    print('bimg in dataloader', bimg.shape)
    with torch.no_grad():
        embedding = sam_model.image_encoder(bimg)
    break


In [ ]:
for name in train_names:
    img_path = os.path.join(path.RAINE_ORGAN_IMAGES_51, name)
    print('image_name', img_path)
    gt_path = os.path.join(path.RAINE_ORGAN_MANUAL_GTS_51, name)
    # load image and gt
    img = sitk.ReadImage(img_path)
    img_array = sitk.GetArrayFromImage(img)
    gt = sitk.ReadImage(gt_path)
    gt_array = sitk.GetArrayFromImage(gt)
    gt_array = np.uint8(gt_array==label_id)
    # norlize image
    img_norm = min_max_norm_3dimg(img_array) 
    break

In [ ]:
z_index = img_array.shape[0] // 2
fig, ax = plt.subplots(2, 2, figsize=(10, 5))
ax[0,0].imshow(img_array[z_index, :, :], cmap='gray')
ax[0,0].set_title('Original Image')
ax[0,0].axis('off')
ax[0,1].imshow(gt_array[z_index, :, :], cmap='gray')
ax[0,1].set_title('Original pancreas GT')
ax[0,1].axis('off')
print('orignal image intensity range', img_array.min(), img_array.max())

z_index_resized = img_norm.shape[0] // 2
ax[1,0].imshow(img_norm[z_index_resized, :, :], cmap='gray')
ax[1,0].set_title('Norm Image')
ax[1,0].axis('off')
ax[1,1].imshow(gt_array[z_index_resized, :, :], cmap='gray')
ax[1,1].set_title('Original pancreas GT')
ax[1,1].axis('off')
print('norm image intensity range', img_norm.min(), img_norm.max())

plt.show()

In [ ]:
# set up the model
sam_model = sam_model_registry[sam_model_type](checkpoint=checkpoint).to(device)

# preprocess the training dataset
# outliers: sub-14870, sub-16880, sub-22770, sub-41810, sub-52220
for name in tqdm(train_names):
    img_path = os.path.join(path.RAINE_ORGAN_IMAGES_51, name)
    gt_path = os.path.join(path.RAINE_ORGAN_MANUAL_GTS_51, name)
    # load image and gt
    img = sitk.ReadImage(img_path)
    img_array = sitk.GetArrayFromImage(img)
    gt = sitk.ReadImage(gt_path)
    gt_array = sitk.GetArrayFromImage(gt)
    gt_array = np.uint8(gt_array==label_id)

    # normalize image
    img_norm = min_max_norm_3dimg(img_array) 

    # preprocess the image and gt for SAM
    imgs, gts, img_embeddings = sam_preprocess(img_norm, gt_array, image_size, sam_model, device)

    # stack the list to array then save to npz file
    if len(imgs)>1:
        imgs = np.stack(imgs, axis=0) # (n, 256, 256, 3)
        gts = np.stack(gts, axis=0) # (n, 256, 256)
        img_embeddings = np.stack(img_embeddings, axis=0) # (n, 1, 256, 64, 64)
        print(name, 'imgs shape', imgs.shape, '\tgts shape', gts.shape)
        np.savez_compressed(os.path.join(save_path_tr, name.split('.nii.gz')[0]+'.npz'), imgs=imgs, gts=gts, img_embeddings=img_embeddings)
        # save an example image for sanity check
        idx = np.random.randint(0, imgs.shape[0])
        img_idx = imgs[idx,:,:,:]
        gt_idx = gts[idx,:,:]
        bd = segmentation.find_boundaries(gt_idx, mode='inner')
        img_idx[bd, :] = [255, 0, 0]
        io.imsave(save_path_tr + '.png', img_idx, check_contrast=False)


In [ ]:
# set up the model
sam_model = sam_model_registry[sam_model_type](checkpoint=checkpoint).to(device)

# preprocess the testing dataset
# outliers: sub-12400
sub_index = {}
for name in tqdm(test_names[:2]):
    img_path = os.path.join(path.RAINE_ORGAN_IMAGES_51, name)
    gt_path = os.path.join(path.RAINE_ORGAN_MANUAL_GTS_51, name)
    # load image and gt
    img = sitk.ReadImage(img_path)
    img_array = sitk.GetArrayFromImage(img)
    gt = sitk.ReadImage(gt_path)
    gt_array = sitk.GetArrayFromImage(gt)
    gt_array = np.uint8(gt_array==label_id)

    # normalize image
    img_norm = min_max_norm_3dimg(img_array) 

    # preprocess the image and gt for SAM
    imgs, gts, img_embeddings, label_index = sam_preprocess(img_norm, gt_array, image_size, sam_model, device)
    sub_index[name] = label_index

    # stack the list to array then save to npz file
    if len(imgs)>1:
        imgs = np.stack(imgs, axis=0) # (n, 256, 256, 3)
        gts = np.stack(gts, axis=0) # (n, 256, 256)
        img_embeddings = np.stack(img_embeddings, axis=0) # (n, 1, 256, 64, 64)
        print(name, 'imgs shape', imgs.shape, '\tgts shape', gts.shape)
        np.savez_compressed(os.path.join(save_path_ts, name.split('.nii.gz')[0]+'.npz'), imgs=imgs, gts=gts, img_embeddings=img_embeddings)
        # save an example image for sanity check
        idx = np.random.randint(0, imgs.shape[0])
        img_idx = imgs[idx,:,:,:]
        gt_idx = gts[idx,:,:]
        bd = segmentation.find_boundaries(gt_idx, mode='inner')
        img_idx[bd, :] = [255, 0, 0]
        io.imsave(save_path_ts + '.png', img_idx, check_contrast=False)

    with open(os.path.join(save_path_ts, 'sub_index.json'), 'w') as f:
        json.dump(sub_index, f, indent=4)


In [ ]:
def custom_formatter(item):
        if isinstance(item, list):
            return json.dumps(item, separators=(',', ':'))
        return json.dumps(item, separators=(',', ':'), indent=4)




In [ ]:
with open(os.path.join(save_path_ts, 'sub_index.json'), 'w') as f:
    json.dump(sub_index, f, indent=4, separators=(', ', ': '))